In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 03b — STAGE 2: Recourse-Absence Confirmation via DECISION-BOUNDARY PROBE
# ═══════════════════════════════════════════════════════════════════════════════
# Stage 1 (Notebook 03) found — using DiCE (random method) + 3 representative cases
# — that Elderly Male IFG→Normal has NO actionable counterfactual (0/324), while
# other cells retain recourse. A referee could object that this reflects a weak
# search method or a tiny sample. Stage 2 removes both objections WITHOUT relying
# on any particular counterfactual generator:
#
#   [PROBE]  For every eligible IFG case we densely sample the ACTIONABLE feature
#            space within realistic training-data bounds (1st–99th percentile) and
#            ask the model directly: does ANY reachable combination yield a Normal
#            prediction (<100 mg/dL)? This maps the model's decision boundary
#            directly, so "no recourse" means "no reachable Normal region exists" —
#            a structural property of the fitted model, not a sampler artefact.
#   [SCALE]  Runs on ALL eligible IFG hold-out cases per group (not a subsample),
#            in seconds, because prediction is vectorised.
#   [DEPTH]  For reachable cases we also report the MINIMUM proximity (nearest
#            reachable Normal combination) — "reachable but how far?" — with
#            Euclidean and Mahalanobis metrics and bootstrap CIs.
#   [CONTRAST] Reported for the same three cells as the DiCE contrast:
#            Elderly Male (absence candidate), Elderly Female (high burden),
#            Young Male (control) — plus the remaining groups for completeness.
#   [IMMUTABLE] Structural variables (income, education, household income,
#            health-screening) are held fixed, exactly as in Stage 1.
#   results/{tables,figures,artifacts}; seaborn grayscale; no captions; dpi=600;
#   png+pdf; no absolute paths printed.
#
# Note: this probe is a SUPERSET test of DiCE recourse — if no densely-sampled
# actionable combination reaches Normal, then no counterfactual method can.
# ═══════════════════════════════════════════════════════════════════════════════

import warnings; warnings.filterwarnings("ignore")
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

SEED = 42; DPI = 600
np.random.seed(SEED)

# ── Stage-2 knobs ─────────────────────────────────────────────────────────────
N_SAMPLES      = 5000      # actionable-space samples drawn per case
TARGET_HI      = 100.0     # Normal threshold (< 100 mg/dL)
BOUND_LO_Q     = 0.01      # actionable sampling bounds: 1st–99th training pctile
BOUND_HI_Q     = 0.99
N_BOOT         = 2000
# groups analysed (all six; the three-way contrast is a subset used in figures)
CONTRAST = ["Elderly_Male", "Elderly_Female", "Young_Male"]

sns.set_theme(style="whitegrid", context="paper"); sns.set_palette("Greys")
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":11,"axes.unicode_minus":False,
                     "figure.dpi":150,"savefig.dpi":DPI,"axes.edgecolor":"0.2","grid.color":"0.85"})
def save_fig(fig, name):
    for ext in ("png","pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "results").is_dir():
            return p
    return start.parent if start.name == "notebooks" else start
ROOT = find_project_root()
FIG_DIR   = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"
ART_DIR   = ROOT / "results" / "artifacts"
for d in (FIG_DIR, TABLE_DIR, ART_DIR): d.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD PRIMITIVES + Stage-1 config
# ═══════════════════════════════════════════════════════════════════════════════
def _load_df_final():
    pq = ART_DIR / "df_final.parquet"; cs = ART_DIR / "df_final.csv"
    if pq.exists():
        try: return pd.read_parquet(pq)
        except Exception: pass
    return pd.read_csv(cs)

df_final    = _load_df_final()
best_models = joblib.load(ART_DIR / "best_models.pkl")
best_algo_name = joblib.load(ART_DIR / "best_algo_name.pkl")
cfg         = joblib.load(ART_DIR / "config.pkl")
try:
    stage1 = joblib.load(ART_DIR / "dice_diagnostics.pkl")
    IMMUTABLE = stage1["IMMUTABLE"]
except Exception:
    IMMUTABLE = ["IncomeQuartile","HouseholdIncome","EducationLevel","HealthScreening"]

X_FEATURES   = cfg["X_FEATURES"]
GROUP_CONFIG = cfg["GROUP_CONFIG"]; GROUP_LABELS = cfg["GROUP_LABELS"]
ACTIONABLE   = [f for f in X_FEATURES if f not in IMMUTABLE]
ACT_IDX      = [X_FEATURES.index(f) for f in ACTIONABLE]

_numc = [c for c in (X_FEATURES+["FPG","AgeGroup","Sex","SurveyYear"]) if c in df_final.columns]
df_final[_numc] = df_final[_numc].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float64")

def assign_stage(fpg):
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"

log.info("Stage 2 (decision-boundary probe) | samples/case=%d | actionable=%d/%d",
         N_SAMPLES, len(ACTIONABLE), len(X_FEATURES))

# ═══════════════════════════════════════════════════════════════════════════════
# 2. PROBE UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════
def feature_stds(df_train):
    s = df_train[X_FEATURES].std().values.astype(float); s[s==0]=1.0; return s
def inv_cov(df_train):
    C = np.cov(df_train[X_FEATURES].values.astype(float).T) + np.eye(len(X_FEATURES))*1e-6
    return np.linalg.pinv(C)

def bootstrap_ci(v, n_boot=N_BOOT, ci=0.95, seed=SEED):
    v=np.asarray(v,float); v=v[~np.isnan(v)]
    if len(v)<2: return (np.nan,np.nan)
    rng=np.random.RandomState(seed)
    b=[np.mean(rng.choice(v,len(v),replace=True)) for _ in range(n_boot)]
    return (round(float(np.percentile(b,(1-ci)/2*100)),4),
            round(float(np.percentile(b,(1+ci)/2*100)),4))

def probe_case(model, base_row, lo, hi, stds, VI, n_samples=N_SAMPLES, rng=None):
    """Densely sample actionable space; return reachability + nearest reachable
       proximity (Euclid/Maha) + the minimum achievable prediction."""
    rng = rng or np.random
    base = base_row[X_FEATURES].values.astype(float)
    M = np.tile(base, (n_samples, 1))
    M[:, ACT_IDX] = rng.uniform(lo, hi, size=(n_samples, len(ACTIONABLE)))
    preds = model.predict(pd.DataFrame(M, columns=X_FEATURES))
    reach_mask = preds < TARGET_HI
    n_reach = int(reach_mask.sum())
    min_pred = float(preds.min())
    if n_reach == 0:
        return {"reachable": False, "n_reach": 0, "min_pred": round(min_pred,2),
                "prox_euclid": np.nan, "prox_maha": np.nan}
    # nearest reachable combination (min Euclidean normalised distance)
    reach = M[reach_mask]
    d = (reach - base) / stds
    de = np.sqrt(np.sum(d*d, axis=1))
    j = int(np.argmin(de))
    diff = reach[j] - base
    pm = float(np.sqrt(max(diff @ VI @ diff, 0.0)))
    return {"reachable": True, "n_reach": n_reach, "min_pred": round(min_pred,2),
            "prox_euclid": round(float(np.mean(np.abs(diff)/stds)),4),
            "prox_maha": round(pm,4)}

# ═══════════════════════════════════════════════════════════════════════════════
# 3. RUN PROBE — ALL eligible IFG hold-out cases, every group
# ═══════════════════════════════════════════════════════════════════════════════
case_rows = []; cell_rows = []
rng = np.random.RandomState(SEED)

for grp, gcfg in GROUP_CONFIG.items():
    model = best_models.get(grp)
    if model is None:
        log.warning("no model for %s — skip", grp); continue
    df_g = df_final[(df_final["AgeGroup"]==gcfg["age_group"]) &
                    (df_final["Sex"]==gcfg["sex_code"])].copy().reset_index(drop=True)
    X, y = df_g[X_FEATURES], df_g["FPG"]
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED)
    df_train = X_train.copy(); df_train["FPG"] = y_train.values
    lo = df_train[ACTIONABLE].quantile(BOUND_LO_Q).values
    hi = df_train[ACTIONABLE].quantile(BOUND_HI_Q).values
    stds = feature_stds(df_train); VI = inv_cov(df_train)

    df_val = X_val.copy(); df_val["FPG"] = y_val.values
    df_val["Stage"] = df_val["FPG"].apply(assign_stage)
    ifg = df_val[df_val["Stage"]=="ifg"].reset_index(drop=True)
    n_elig = len(ifg)
    log.info("── %s : IFG→Normal | eligible=%d (probing all) ──", GROUP_LABELS[grp], n_elig)
    if n_elig == 0:
        cell_rows.append({"Group":GROUP_LABELS[grp], "_key":grp, "n_eligible":0,
            "n_reachable":0, "reachable_rate":np.nan, "median_min_pred":np.nan})
        continue

    n_reach_cell = 0; min_preds = []
    for i, (_, row) in enumerate(ifg.iterrows()):
        r = probe_case(model, row, lo, hi, stds, VI, rng=rng)
        n_reach_cell += int(r["reachable"]); min_preds.append(r["min_pred"])
        case_rows.append({"Group":GROUP_LABELS[grp], "_key":grp, "case":i+1,
            "orig_fpg":round(float(row["FPG"]),1), **r})
    rate = round(n_reach_cell/n_elig, 3)
    cell_rows.append({"Group":GROUP_LABELS[grp], "_key":grp, "n_eligible":n_elig,
        "n_reachable":n_reach_cell, "reachable_rate":rate,
        "median_min_pred":round(float(np.median(min_preds)),2)})
    log.info("  >>> reachable Normal in %d/%d (rate=%.3f) | median min-pred=%.1f",
             n_reach_cell, n_elig, rate, float(np.median(min_preds)))

cases = pd.DataFrame(case_rows)
cells = pd.DataFrame(cell_rows)

print("\n── [Stage 2 PROBE] Reachability of Normal by group (IFG→Normal, all eligible) ──")
print(cells.drop(columns=["_key"]).to_string(index=False))
cells.drop(columns=["_key"]).to_csv(TABLE_DIR/"stage2_reachability.csv", index=False, encoding="utf-8")
cases.drop(columns=["_key"]).to_csv(TABLE_DIR/"stage2_case_probe.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. PROXIMITY where reachable (nearest reachable Normal), with bootstrap CI
# ═══════════════════════════════════════════════════════════════════════════════
prox_rows = []
for grp in GROUP_CONFIG:
    lab = GROUP_LABELS[grp]
    got = cases[(cases["Group"]==lab) & (cases["reachable"])]
    for metric in ["prox_euclid","prox_maha"]:
        vals = got[metric].dropna().values
        if len(vals)==0:
            prox_rows.append({"Group":lab,"Metric":metric,"n":0,"mean":np.nan,"CI_lo":np.nan,"CI_hi":np.nan})
        else:
            lo_,hi_ = bootstrap_ci(vals)
            prox_rows.append({"Group":lab,"Metric":metric,"n":len(vals),
                "mean":round(float(np.mean(vals)),4),"CI_lo":lo_,"CI_hi":hi_})
prox_df = pd.DataFrame(prox_rows)
print("\n── [Stage 2 PROBE] Nearest-reachable proximity where recourse exists ──")
print(prox_df.to_string(index=False))
prox_df.to_csv(TABLE_DIR/"stage2_proximity.csv", index=False, encoding="utf-8")

# Ratios vs Young Male (Euclid + Maha) with bootstrap CI on the ratio
def ratio_ci(a,b,n_boot=N_BOOT,seed=SEED):
    a=np.asarray(a,float); a=a[~np.isnan(a)]; b=np.asarray(b,float); b=b[~np.isnan(b)]
    if len(a)==0 or len(b)==0: return (np.nan,np.nan,np.nan)
    rng_=np.random.RandomState(seed); rs=[]
    for _ in range(n_boot):
        rs.append(np.mean(rng_.choice(a,len(a),replace=True))/max(np.mean(rng_.choice(b,len(b),replace=True)),1e-9))
    return (round(float(np.mean(a))/max(float(np.mean(b)),1e-9),2),
            round(float(np.percentile(rs,2.5)),2), round(float(np.percentile(rs,97.5)),2))

def vals_of(lab, metric):
    return cases[(cases["Group"]==lab)&(cases["reachable"])][metric].dropna().values
ratio_rows=[]
for metric in ["prox_euclid","prox_maha"]:
    ym = vals_of("Young Male", metric)
    for lab in ["Elderly Female","Elderly Male","Middle-aged Male","Middle-aged Female","Young Female"]:
        arr = vals_of(lab, metric); r = ratio_ci(arr, ym)
        ratio_rows.append({"Metric":metric, "Comparison":f"{lab} / Young Male",
            "Ratio":r[0], "CI_lo":r[1], "CI_hi":r[2],
            "note":("undefined — no reachable recourse" if len(arr)==0 else "")})
ratio_df = pd.DataFrame(ratio_rows)
print("\n── [Stage 2 PROBE] Proximity ratios vs Young Male (95% bootstrap CI) ──")
print(ratio_df.to_string(index=False))
ratio_df.to_csv(TABLE_DIR/"stage2_ratios.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. FIGURES (grayscale; no captions; dpi=600; png+pdf)
# ═══════════════════════════════════════════════════════════════════════════════
order6 = [GROUP_LABELS[g] for g in GROUP_CONFIG]

# Fig A: reachability rate by group (headline)
cc = cells.set_index("Group").reindex(order6)
fig, ax = plt.subplots(figsize=(9,4.5))
ax.bar(cc.index, cc["reachable_rate"].fillna(0), color="0.55", edgecolor="black")
for i,(g,r) in enumerate(cc.iterrows()):
    if not np.isnan(r["reachable_rate"]):
        ax.text(i, (r["reachable_rate"] or 0)+0.02,
                f"{r['reachable_rate']:.2f}\n({int(r['n_reachable'])}/{int(r['n_eligible'])})",
                ha="center", va="bottom", fontsize=8)
ax.set_ylim(0,1.15); ax.set_ylabel("Reachable-Normal rate (IFG→Normal)")
ax.set_xlabel(""); ax.set_xticklabels(order6, rotation=25, ha="right", fontsize=8)
plt.tight_layout(); save_fig(fig, "fig_stage2_reachability")

# Fig B: distribution of minimum achievable prediction per group (shows the wall)
if len(cases):
    cc2 = cases.copy()
    cc2["Group"] = pd.Categorical(cc2["Group"], categories=order6, ordered=True)
    fig, ax = plt.subplots(figsize=(9,4.5))
    sns.boxplot(data=cc2, x="Group", y="min_pred", color="0.7", fliersize=2, linewidth=1.0, ax=ax)
    ax.axhline(TARGET_HI, ls="--", color="0.2", lw=1.2)   # Normal threshold
    ax.set_xlabel(""); ax.set_ylabel("Minimum achievable FPG prediction (mg/dL)")
    ax.set_xticklabels(order6, rotation=25, ha="right", fontsize=8)
    plt.tight_layout(); save_fig(fig, "fig_stage2_min_pred")

# Fig C: nearest-reachable proximity (Euclid) where recourse exists
got = cases[cases["reachable"]].copy()
if len(got):
    got["Group"] = pd.Categorical(got["Group"], categories=order6, ordered=True)
    fig, ax = plt.subplots(figsize=(9,4.5))
    sns.boxplot(data=got, x="Group", y="prox_euclid", color="0.7", fliersize=2, linewidth=1.0, ax=ax)
    sns.stripplot(data=got, x="Group", y="prox_euclid", color="0.2", size=2, alpha=0.35, ax=ax)
    ax.set_xlabel(""); ax.set_ylabel("Nearest-reachable Euclidean proximity")
    ax.set_xticklabels(order6, rotation=25, ha="right", fontsize=8)
    plt.tight_layout(); save_fig(fig, "fig_stage2_proximity_box")

log.info("Figures saved (png + pdf, dpi=%d)", DPI)

# ═══════════════════════════════════════════════════════════════════════════════
# 6. SAVE + STAGE-2 VERDICT
# ═══════════════════════════════════════════════════════════════════════════════
joblib.dump({"cells":cells, "cases":cases, "prox_df":prox_df, "ratio_df":ratio_df,
             "config":{"method":"decision-boundary probe","n_samples":N_SAMPLES,
                       "target_hi":TARGET_HI, "bounds_q":(BOUND_LO_Q,BOUND_HI_Q),
                       "immutable":IMMUTABLE, "actionable":ACTIONABLE}},
            ART_DIR / "dice_stage2.pkl")

print("\n"+"="*72); print("STAGE-2 VERDICT — decision-boundary probe (%d samples/case, all eligible)"%N_SAMPLES); print("="*72)
for _, r in cells.iterrows():
    if r["n_eligible"]==0: continue
    rate = r["reachable_rate"]
    tag = ("✗ RECOURSE ABSENT" if rate==0.0 else
           "△ recourse RARE" if rate<0.5 else "✓ recourse available")
    extra = f" | median min-pred={r['median_min_pred']:.1f} (>=100 ⇒ wall above Normal)" if rate==0.0 else ""
    print(f"  {tag:20s} {r['Group']:18s} {int(r['n_reachable'])}/{int(r['n_eligible'])} "
          f"(rate={rate}){extra}")
print("\n  A rate of 0 with median min-pred >= 100 means: across %d dense actionable" % N_SAMPLES)
print("  samples per case, the model NEVER predicts Normal — a wall in the decision")
print("  boundary. No counterfactual method (DiCE random/genetic/gradient) can cross it.")
log.info("Notebook 03b (Stage 2 probe) complete.")

2026-09-01 16:24:51,512 | INFO | Stage 2 (decision-boundary probe) | samples/case=5000 | actionable=27/31
2026-09-01 16:24:51,541 | INFO | ── Young Male : IFG→Normal | eligible=125 (probing all) ──
2026-09-01 16:24:54,546 | INFO |   >>> reachable Normal in 125/125 (rate=1.000) | median min-pred=89.3
2026-09-01 16:24:54,569 | INFO | ── Young Female : IFG→Normal | eligible=70 (probing all) ──
2026-09-01 16:24:55,662 | INFO |   >>> reachable Normal in 70/70 (rate=1.000) | median min-pred=81.0
2026-09-01 16:24:55,701 | INFO | ── Middle-aged Male : IFG→Normal | eligible=430 (probing all) ──
2026-09-01 16:25:02,773 | INFO |   >>> reachable Normal in 429/430 (rate=0.998) | median min-pred=94.5
2026-09-01 16:25:02,809 | INFO | ── Middle-aged Female : IFG→Normal | eligible=409 (probing all) ──
2026-09-01 16:25:09,492 | INFO |   >>> reachable Normal in 408/409 (rate=0.998) | median min-pred=90.8
2026-09-01 16:25:09,514 | INFO | ── Elderly Male : IFG→Normal | eligible=324 (probing all) ──
2026-09


── [Stage 2 PROBE] Reachability of Normal by group (IFG→Normal, all eligible) ──
             Group  n_eligible  n_reachable  reachable_rate  median_min_pred
        Young Male         125          125           1.000            89.32
      Young Female          70           70           1.000            81.03
  Middle-aged Male         430          429           0.998            94.53
Middle-aged Female         409          408           0.998            90.80
      Elderly Male         324            0           0.000           103.42
    Elderly Female         347          347           1.000            93.78

── [Stage 2 PROBE] Nearest-reachable proximity where recourse exists ──
             Group      Metric   n    mean   CI_lo   CI_hi
        Young Male prox_euclid 125  0.7987  0.7642  0.8426
        Young Male   prox_maha 125 13.6683 13.1939 14.1763
      Young Female prox_euclid  70  0.8049  0.7670  0.8480
      Young Female   prox_maha  70 14.1178 13.3276 14.9659
  Middle-ag

2026-09-01 16:26:00,404 | INFO | Figures saved (png + pdf, dpi=600)
2026-09-01 16:26:00,425 | INFO | Notebook 03b (Stage 2 probe) complete.



STAGE-2 VERDICT — decision-boundary probe (5000 samples/case, all eligible)
  ✓ recourse available Young Male         125/125 (rate=1.0)
  ✓ recourse available Young Female       70/70 (rate=1.0)
  ✓ recourse available Middle-aged Male   429/430 (rate=0.998)
  ✓ recourse available Middle-aged Female 408/409 (rate=0.998)
  ✗ RECOURSE ABSENT    Elderly Male       0/324 (rate=0.0) | median min-pred=103.4 (>=100 ⇒ wall above Normal)
  ✓ recourse available Elderly Female     347/347 (rate=1.0)

  A rate of 0 with median min-pred >= 100 means: across 5000 dense actionable
  samples per case, the model NEVER predicts Normal — a wall in the decision
  boundary. No counterfactual method (DiCE random/genetic/gradient) can cross it.
